# Imports y rutas

In [ ]:
# importo las librerías necesarias para todo el pipeline de detección con yolo
import os
import numpy as np
import pandas as pd
import shutil
import yaml
from PIL import Image
from sklearn.model_selection import train_test_split

# defino las rutas base del entorno kaggle
RUTA_DATOS      = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge'
# carpeta con los PNGs de train ya convertidos
RUTA_PNG_TRAIN  = '/kaggle/input/datasets/franciscofdzfer/rsna-png-512/png_train'
# carpeta con los PNGs de test ya convertidos
RUTA_PNG_TEST   = '/kaggle/input/datasets/franciscofdzfer/rsna-png-512-test/png_test'
# carpeta de salida para labels, dataset y modelos
RUTA_OUTPUTS    = '/kaggle/working'

print('rutas configuradas')

# Instalar Ultralytics

In [ ]:
# instalo ultralytics para usar yolov11 con una sola línea de entrenamiento
!pip install ultralytics --quiet

# Cargar labels

In [ ]:
# cargo el csv de labels para construir las anotaciones en formato yolo
df_labels = pd.read_csv(f'{RUTA_DATOS}/stage_2_train_labels.csv')

# muestro distribución básica
print(df_labels.shape)
print(df_labels['Target'].value_counts())

# Celda 4 — Verificar espacio disponible

In [ ]:
# verifico el espacio disponible antes de generar labels para no quedarnos sin disco
total, usado, libre = shutil.disk_usage('/kaggle/working')
print(f'total:  {total  / (1024**3):.1f} GB')
print(f'usado:  {usado  / (1024**3):.1f} GB')
print(f'libre:  {libre  / (1024**3):.1f} GB')

# Crear estructura de carpetas YOLO

In [ ]:
# creo la estructura de carpetas que yolo espera para encontrar imágenes y labels
carpetas = [
    f'{RUTA_OUTPUTS}/dataset/images/train',
    f'{RUTA_OUTPUTS}/dataset/images/val',
    f'{RUTA_OUTPUTS}/dataset/labels/train',
    f'{RUTA_OUTPUTS}/dataset/labels/val',
]

for carpeta in carpetas:
    # crea la carpeta si no existe
    os.makedirs(carpeta, exist_ok=True)

print('estructura de carpetas creada')

# split

In [ ]:
# elimino pacientes repetidos y divio los datos en 85% para entrenar y 15% para evaluar
# manteniendo igual proporción de enfermos

# construyo un df único por paciente y hago el split estratificado antes de procesar imágenes
df_unique = df_labels.drop_duplicates(subset='patientId')[['patientId', 'Target']]

# split 85/15 estratificado por clase
df_train, df_val = train_test_split(
    df_unique,
    test_size=0.15,
    random_state=42,
    stratify=df_unique['Target']
)

print(f'train: {len(df_train)} — positivos: {df_train["Target"].sum()}')
print(f'val:   {len(df_val)}   — positivos: {df_val["Target"].sum()}')

# Función para generar labels formato YOLO

In [ ]:
# Crea archivos de texto con la ubicación exacta de la enfermedad
# adaptando las medidas al tamaño y formato que exige YOLO

# convierto las coordenadas absolutas del csv al formato yolo normalizado [0-1]
def generar_label_yolo(patient_id, split):
    # obtengo todas las filas del paciente
    filas = df_labels[df_labels['patientId'] == patient_id]
    # ruta donde se guardará el txt
    ruta_txt = f'{RUTA_OUTPUTS}/dataset/labels/{split}/{patient_id}.txt'

    # si no hay neumonía creo un txt vacío
    if filas['Target'].iloc[0] == 0:
        open(ruta_txt, 'w').close()
        return

    # escribo una línea por cada bounding box
    # para que el modelo sepa cuántas zonas enfermas hay en una misma radiografía y dónde está cada una
    with open(ruta_txt, 'w') as f:
        for _, fila in filas.iterrows():
            if fila['Target'] == 1:
                # tamaño original de las imágenes dicom
                tam = 1024
                # convierto a coordenadas centradas normalizadas
                cx = (fila['x'] + fila['width']  / 2) / tam
                cy = (fila['y'] + fila['height'] / 2) / tam
                w  = fila['width']  / tam
                h  = fila['height'] / tam
                # clase 0 = neumonía
                f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

# Test función label sobre 5 pacientes

In [ ]:
# pruebo la función sobre 5 pacientes antes de lanzar el bucle completo
ids_test_label = df_train['patientId'].values[:5]

for pid in ids_test_label:
    generar_label_yolo(pid, 'train')

# muestro el contenido de los labels generados
# las coordenadas finales calculadas para sus zonas enfermas
for pid in ids_test_label:
    ruta_txt = f'{RUTA_OUTPUTS}/dataset/labels/train/{pid}.txt'
    contenido = open(ruta_txt).read()
    print(f'{pid[:8]}: "{contenido.strip()}"')

# Generar todos los labels train y val

In [ ]:
# genero los labels para todos los pacientes de train y val

from tqdm.notebook import tqdm

for pid in tqdm(df_train['patientId'], desc='labels train'):
    generar_label_yolo(pid, 'train')

for pid in tqdm(df_val['patientId'], desc='labels val'):
    generar_label_yolo(pid, 'val')

# verifico el número de labels generados
n_train = len(os.listdir(f'{RUTA_OUTPUTS}/dataset/labels/train'))
n_val   = len(os.listdir(f'{RUTA_OUTPUTS}/dataset/labels/val'))
print(f'labels train: {n_train}')
print(f'labels val:   {n_val}')

# Copiar imágenes PNG a la estructura YOLO

In [ ]:
# copio los PNGs a las carpetas de yolo enlazando cada imagen con su label

# copio las radiografías reales en sus respectivas carpetas de entrenamiento y validación
# y cuenta cuántas imágenes se guardaron en total
for pid in tqdm(df_train['patientId'], desc='imágenes train'):
    src = f'{RUTA_PNG_TRAIN}/{pid}.png'
    dst = f'{RUTA_OUTPUTS}/dataset/images/train/{pid}.png'
    shutil.copy(src, dst)

for pid in tqdm(df_val['patientId'], desc='imágenes val'):
    src = f'{RUTA_PNG_TRAIN}/{pid}.png'
    dst = f'{RUTA_OUTPUTS}/dataset/images/val/{pid}.png'
    shutil.copy(src, dst)

# verifico el número de imágenes copiadas
n_train = len(os.listdir(f'{RUTA_OUTPUTS}/dataset/images/train'))
n_val   = len(os.listdir(f'{RUTA_OUTPUTS}/dataset/images/val'))
print(f'imágenes train: {n_train}')
print(f'imágenes val:   {n_val}')

# Generar data.yaml

In [ ]:
# genero el archivo de configuración que yolo necesita para encontrar los datos y las clases

# creo el archivo "data_yaml" de configuración que le dice a YOLO dónde buscar 
# las fotos y que solo debe buscar una clase: "pneumonía"
data_yaml = {
    'path'  : f'{RUTA_OUTPUTS}/dataset',
    'train' : 'images/train',
    'val'   : 'images/val',
    # lo que le avisa a YOLO que solo buscará un tipo de objeto
    'nc'    : 1,
    # le digo que la clase con el índice se llama 'pneumonia'
    'names' : ['pneumonia']
}

ruta_yaml = f'{RUTA_OUTPUTS}/dataset/data.yaml'
with open(ruta_yaml, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

# muestro el contenido del yaml generado
print(open(ruta_yaml).read())

# Test con 200 imágenes

In [ ]:
# pruebo el pipeline completo con 200 imágenes antes de lanzar el entrenamiento completo
from ultralytics import YOLO

# cargo el modelo yolov11m preentrenado
# carga los pesos preentrenados de la versión "Medium" (mediana) de YOLO11
# al estar preentrenado ya sabe detectar formas básica, bordes y estructuras.
modelo_test = YOLO('yolo11m.pt')

# entreno solo 2 epochs sobre 200 imágenes para verificar que no hay errores
resultado_test = modelo_test.train(
    data    = ruta_yaml,
    epochs  = 2,
    imgsz   = 512,
    batch   = 16,
    # utilicen las 2 GPUs
    device  = '0,1',
    # YOLO use solo 200 imágenes al azar
    fraction= 200 / len(df_train),
    project = f'{RUTA_OUTPUTS}/test_run',
    name    = 'yolo_test',
    exist_ok= True,
    verbose = False
)

print('test completado sin errores')

# Entrenamiento completo YOLOv11m

In [ ]:
# verifico espacio antes de lanzar el entrenamiento completo
total, usado, libre = shutil.disk_usage('/kaggle/working')
print(f'libre: {libre / (1024**3):.1f} GB')
# verifico memoria gpu disponible
import torch
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB')

# entreno yolov11m con todas las imágenes durante 30 epochs guardando el mejor modelo
from ultralytics import YOLO
modelo = YOLO('yolo11m.pt')
resultado = modelo.train(
    data     = ruta_yaml,
    epochs   = 30,
    imgsz    = 512,
    batch    = 16,
    device   = '0,1',
    project  = f'{RUTA_OUTPUTS}/runs',
    name     = 'yolo_rsna',
    exist_ok = True,
    flipud   = 0.0,
    fliplr   = 0.5,
    hsv_h    = 0.0,
    hsv_s    = 0.0,
    hsv_v    = 0.2,
    degrees  = 0.0,
    patience = 10,
)
print(f'entrenamiento completado')
print(f'mejor modelo: {RUTA_OUTPUTS}/runs/yolo_rsna/weights/best.pt')

# Guardar modelo como dataset privado

In [ ]:
# guardo el modelo como dataset privado para no perderlo si se cierra la sesión
import subprocess
import json
import shutil

ruta_modelo_dir = f'{RUTA_OUTPUTS}/modelo_guardado'
os.makedirs(ruta_modelo_dir, exist_ok=True)
shutil.copy(f'{RUTA_OUTPUTS}/runs/yolo_rsna/weights/best.pt', ruta_modelo_dir)

metadata = {
    "title": "rsna-yolo-best",
    "id": "franciscofdzfer/rsna-yolo-best",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(f'{ruta_modelo_dir}/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

subprocess.run([
    'kaggle', 'datasets', 'create',
    '-p', ruta_modelo_dir,
    '--dir-mode', 'zip'
])
print('modelo guardado como dataset privado')

# Inferencia sobre test y submission

In [ ]:
# genero la submission corrigiendo coordenadas de centro a esquina superior izquierda
from ultralytics import YOLO
import pandas as pd
import os

RUTA_MODELO   = '/kaggle/input/datasets/franciscofdzfer/rsna-yolo-best/best.pt'
RUTA_PNG_TEST = '/kaggle/input/datasets/franciscofdzfer/rsna-png-512-test/png_test'
RUTA_OUTPUT   = '/kaggle/working'

mejor_modelo = YOLO(RUTA_MODELO)
ids_test     = [f.replace('.png', '') for f in os.listdir(RUTA_PNG_TEST) if f.endswith('.png')]
print(f'modelo cargado | imágenes de test: {len(ids_test)}')

resultados = []
for pid in ids_test:
    ruta_png = f'{RUTA_PNG_TEST}/{pid}.png'
    preds    = mejor_modelo.predict(ruta_png, conf=0.3, verbose=False)
    pred     = preds[0]

    if len(pred.boxes) == 0:
        pred_str = ''
    else:
        partes = []
        for box in pred.boxes:
            conf         = box.conf.item()
            cx, cy, w, h = box.xywh[0].tolist()
            # convierto centro a esquina superior izquierda antes de escalar
            x_min = int((cx - w / 2) * (1024 / 512))
            y_min = int((cy - h / 2) * (1024 / 512))
            w_esc = int(w * (1024 / 512))
            h_esc = int(h * (1024 / 512))
            partes.append(f'{conf:.4f} {x_min} {y_min} {w_esc} {h_esc}')
        pred_str = ' '.join(partes)

    resultados.append({'patientId': pid, 'PredictionString': pred_str})

df_sub = pd.DataFrame(resultados)
df_sub.to_csv(f'{RUTA_OUTPUT}/submission_yolo.csv', index=False)
print(f'submission generada: {len(df_sub)} filas')
print(f'con detecciones:     {(df_sub["PredictionString"] != "").sum()}')
print(f'sin detecciones:     {(df_sub["PredictionString"] == "").sum()}')
df_sub[df_sub['PredictionString'] != ''].head(3)

# notificaciones a Discord

In [ ]:
# notifico a discord cuando la submission está generada y lista para subir
import requests

WEBHOOK_URL = 'https://discord.com/api/webhooks/1519666190845218917/8vD-szvGUt3BGevELU3vxGZ1tox5hSDwjtpYQkrJg5h2eb2nRSdTgW-oOClXcn03MVHS'

def notificar_discord(mensaje):
    # envía el mensaje al canal de discord configurado
    requests.post(WEBHOOK_URL, json={'content': mensaje})

if os.path.exists(f'{RUTA_OUTPUTS}/submission_yolo.csv'):
    notificar_discord(
        f'✅ RSNA Fase 3 completada\n'
        f'📊 Submission generada: {len(df_sub)} filas\n'
        f'📁 Modelo en: {RUTA_OUTPUTS}/runs/yolo_rsna/weights/best.pt'
    )
else:
    notificar_discord('❌ Error — submission_yolo.csv no se generó')

# Apagar kernel

In [ ]:
# apago el kernel al terminar para no consumir horas de gpu innecesarias
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)